# STGF-LLM

In [1]:
import os
import pandas as pd
import numpy as np
import torch
import shutil
from datasets import Dataset 
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig, 
    get_peft_model, 
    prepare_model_for_kbit_training,
    PeftModel
)
from tqdm import tqdm
import re
import gc


# Verify GPU is detected
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")



GPU Available: True
GPU Count: 1


Configuration of the base model, and the quantization to 4-bit.

In [2]:
BASE_MODEL = "Qwen/Qwen3-1.7B" 

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Loading Base Model into Memory...")
# Quantize the base model weights to 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load the tokenizer of the base model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load the base model quantized in 4-bit
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={"": 0},
    dtype=torch.float16,
    low_cpu_mem_usage=True
)

# Saving memory by forget the middle steps (activations) of computation and re-calculate them only when needed (backpropagation)
base_model.gradient_checkpointing_enable() 
# De-quantizes the specific layers (LayerNorm and Output) to do computation with LoRA layers
base_model = prepare_model_for_kbit_training(base_model) 
# Enables gradient tracking on the input embeddings
base_model.enable_input_require_grads()
# Set to false in the training step
base_model.config.use_cache = False


# Define Dataset configuration to handle different datasets
class DatasetConfig:
    def __init__(self, name, csv_path, window_size, output_adapter_dir, token_limit):
        self.name = name
        self.csv_path = csv_path
        self.window_size = window_size
        self.output_dir = output_adapter_dir
        self.token_limit = token_limit


dataset_stocks = DatasetConfig(
    name="GOOGL_Stocks", 
    csv_path="./data/GOOGL_STOCKS/GOOGL_STOCKS_llm_text.csv", 
    window_size=10, 
    output_adapter_dir="./stgf_llm_adapters/stocks_lora",
    token_limit=80
)

dataset_occupancy = DatasetConfig(
    name="OccupancyDetection", 
    csv_path="./data/OccupancyDetection/OccupancyDetection_llm_text.csv", 
    window_size=10, 
    output_adapter_dir="./stgf_llm_adapters/occupancy_lora",
    token_limit=85
)

dataset_power = DatasetConfig(
    name="PowerConsumption", 
    csv_path="./data/PowerConsumption/PowerConsumption_llm_text.csv", 
    window_size=10, 
    output_adapter_dir="./stgf_llm_adapters/power_lora",
    token_limit=135
)

dataset_traffic = DatasetConfig(
    name="TrafficVolume", 
    csv_path="./data/TrafficVolume/TrafficVolume_llm_text.csv", 
    window_size=10, 
    output_adapter_dir="./stgf_llm_adapters/traffic_lora",
    token_limit=75
)

Loading Base Model into Memory...


c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\modeling_utils.py:6105: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.empty(byte_count // factor, dtype=torch.float16, device=device, requires_grad=False)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Helper functions

In [3]:
PROMPT_SEPARATOR = "Generate Next Step:"

def create_hf_dataset(windows, prompting_func):
    data_list = []
    SEP = " | " 
    
    if len(windows) > 0:
        sample_row = windows[0][0]
        columns_list = list(sample_row.keys())
        schema_header = f"Dataset Columns: {', '.join(columns_list)}"
    else:
        schema_header = ""

    for window in windows:
        history_rows = window[:-1]
        target_row = window[-1]
        
        history_str_list = [prompting_func(row.copy()) for row in history_rows]
        history_text = SEP.join(history_str_list)
        target_text = prompting_func(target_row.copy())

        full_text = (
            f"{schema_header}\n"
            f"History: {history_text}\n"
            f"{PROMPT_SEPARATOR} {target_text}"
        )
        data_list.append({"text": full_text})

    return Dataset.from_list(data_list)

def strategy_get_prompting(row_dict):
    items = [(k, v) for k, v in row_dict.items() if pd.notna(v)]
    pairs = [f"{k}={v}" for k, v in items]
    return ", ".join(pairs)

def parse_model_output(full_text, expected_cols):
    parsed_data = {}
    
    if PROMPT_SEPARATOR in full_text:
        prediction_str = full_text.split(PROMPT_SEPARATOR)[1]
    else:
        prediction_str = full_text[-300:] 

    pattern = re.compile(r"([\w\s]+)=([^,|]+)")
    matches = pattern.findall(prediction_str)
    
    found_keys = set()
    for key, val_str in matches:
        clean_key = key.strip()
        
        if "Date" in clean_key:
            clean_val = val_str.strip()
        else:
            clean_val = val_str.split(':')[0].split('=')[0].strip()
        
        if clean_key in expected_cols and clean_key not in found_keys:
            if clean_val:
                found_keys.add(clean_key)
                if "Date" in clean_key:
                    parsed_data[clean_key] = clean_val
                elif "weather_main" in clean_key:
                    parsed_data[clean_key] = clean_val
                else:
                    try:
                        numeric_val = re.sub(r'[^\d.]', '', clean_val)
                        if numeric_val == "" or numeric_val == ".": continue
                        if numeric_val.count('.') > 1: continue 
                        parsed_data[clean_key] = float(numeric_val)
                    except ValueError:
                        pass
    return parsed_data

def zip_folder(folder_path):
    shutil.make_archive(folder_path, 'zip', folder_path)

Training the dataset with a specific adapter.

In [4]:
def train_specific_adapter(global_model, config, permutation_func):
    if isinstance(global_model, PeftModel):
        print(f"!!! Warning: Model still has an adapter attached. Unloading now...")
        global_model = global_model.unload() 
    if hasattr(global_model, "unload"):
        global_model = global_model.unload()
    if hasattr(global_model, "peft_config"):
        print("Cleaning lingering PEFT config...")
        del global_model.peft_config
        global_model.config.use_cache = False

    print(f"\n--- Starting Training for: {config.name} ---")

    peft_config = LoraConfig(
        r=32, # rank: the size of the bottleneck in the adapter (shared dimension betwwen A and B)
        lora_alpha=64, # scaling factor: strenght of the adapter updates
        lora_dropout=0.05, # memory loss: regularization to prevent overfitting
        bias="none", 
        task_type="CAUSAL_LM", # casual: predict the next token based on previous token
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"] # linear layers in different modules (attention, MLP)
    )
    
    # Get the model with LoRA configuration, frozen all the base model weights
    model = get_peft_model(global_model, peft_config)
    model.config.use_cache = False
    model.print_trainable_parameters()


    df = pd.read_csv(config.csv_path).round(2)
    records = df.to_dict('records')
    windows = [records[i : i + config.window_size + 1] for i in range(len(records) - config.window_size)]
    
    # Generate the prompt
    raw_dataset = create_hf_dataset(windows, permutation_func)
    
    # Dynamic Max Length Calculation
    lengths = [len(tokenizer.encode(x['text'])) for x in raw_dataset]
    final_max_length = min(max(lengths) + 20, 2048) 
    print(f"Max Sequence Length found: {max(lengths)}")
    print(f"Training using Max Length: {final_max_length}")

    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=final_max_length,
            padding="max_length",
        )
    
    # Tokenize the prompt into tokens
    tokenized_dataset = raw_dataset.map(tokenize_function, batched=True, remove_columns=raw_dataset.column_names)

    training_args = TrainingArguments(
        output_dir=config.output_dir,
        per_device_train_batch_size=4,      
        gradient_accumulation_steps=4, 
        num_train_epochs=2,
        learning_rate=2e-4,
        weight_decay=0.01,
        fp16=True,
        bf16=False,
        logging_steps=20,
        save_strategy="no",
        report_to="none",
        gradient_checkpointing=True,      
        dataloader_num_workers=2,
        optim="adamw_torch",
    )
    
    trainer = Trainer(
        model=model,
        train_dataset=tokenized_dataset,
        args=training_args,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
        processing_class=tokenizer
    )
    
    print("Starting training...")
    trainer.train()
    
    
    print(f"Saving Adapter to {config.output_dir}...")
    model.save_pretrained(config.output_dir)
    zip_folder(config.output_dir)

    print("Unloading adapter and clearing GPU memory...")
    model = model.unload() 
    del trainer
    del model
    gc.collect() 
    torch.cuda.empty_cache()
    torch.cuda.synchronize() 
    print(f"--- Finished: {config.name}. Memory cleared. ---\n")

Generate/predict the synthetic or forecasting data with the adapter associated to a specific dataset.

In [5]:
def generate_with_adapter(global_model, config, target_rows=None, forecasting=False):
    if isinstance(global_model, PeftModel):
        global_model = global_model.unload()
    if hasattr(global_model, "unload"):
        global_model = global_model.unload()
    if hasattr(global_model, "peft_config"):
        del global_model.peft_config

    print(f"Loading adapter from {config.output_dir}...")
    model = PeftModel.from_pretrained(global_model, config.output_dir, is_trainable=False)
    model.eval()

    # Load Data
    df_real = pd.read_csv(config.csv_path).round(2)
    expected_keys = df_real.columns.tolist()
    
    if forecasting:
        real_seed_df = df_real.iloc[-config.window_size:]
    else:
        real_seed_df = df_real.iloc[:config.window_size]
    if target_rows is None: target_rows = len(df_real)

    current_history = real_seed_df.to_dict('records')
    synthetic_rows = []
    
    print(f"Generating {target_rows} rows for {config.name}...")

    # Start the inference mode
    with torch.inference_mode():
        for i in tqdm(range(target_rows)):
            
            history_block = " | ".join([
                ", ".join([f"{k}={v}" for k, v in row.items() if pd.notna(v)])
                for row in current_history
            ])
            
            prompt = f"Dataset Columns: {', '.join(expected_keys)}\nHistory: {history_block}\n{PROMPT_SEPARATOR} "
            
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

            dynamic_temp = 0.1 if forecasting else 0.7
            
            gen_kwargs = {
                "max_new_tokens": config.token_limit,
                "do_sample": True,
                "temperature": dynamic_temp,
                "top_p": 0.9,
                "repetition_penalty": 1.15,
                "pad_token_id": tokenizer.eos_token_id,
                "use_cache": True
            }
            
            outputs = model.generate(**inputs, **gen_kwargs)
            full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            new_row = parse_model_output(full_text, expected_keys)
            
            # Fallback Logic
            last_row = current_history[-1]
            final_row = {}
            for k in expected_keys:
                if k in new_row:
                    final_row[k] = new_row[k]
                else:
                    old_val = last_row.get(k, 0)
                    if isinstance(old_val, (int, float)):
                        final_row[k] = old_val 
                    else:
                        final_row[k] = old_val

            # print(full_text)
            # print(new_row)
            # print(final_row)
            synthetic_rows.append(final_row)
            current_history.pop(0)
            current_history.append(final_row)

    if forecasting:
        output_path = config.csv_path.replace(
            "llm_text.csv",
            "stgf_llm_forecasting.csv"
        )
    else:
        output_path = config.csv_path.replace(
            "llm_text.csv",
            "stgf_llm_synthetic.csv"
        )
    pd.DataFrame(synthetic_rows).to_csv(output_path, index=False)
    print(f"Done! Saved to {output_path}")
    
    model.unload()
    torch.cuda.empty_cache()

Generate synthetic and forecast data.

In [ ]:
stock_test_len = len(pd.read_csv("./data/GOOGL_STOCKS/GOOGL_STOCKS_test.csv"))
occupancy_test_len = len(pd.read_csv("./data/OccupancyDetection/OccupancyDetection_test.csv"))
power_test_len = len(pd.read_csv( "./data/PowerConsumption/PowerConsumption_test.csv"))
traffic_test_len = len(pd.read_csv( "./data/TrafficVolume/TrafficVolume_test.csv"))


# 1. Stocks
train_specific_adapter(base_model, dataset_stocks, strategy_get_prompting)
generate_with_adapter(base_model, dataset_stocks)
generate_with_adapter(base_model, dataset_stocks, target_rows=stock_test_len, forecasting=True)


# 2. Occupancy Detection
train_specific_adapter(base_model, dataset_occupancy, strategy_get_prompting)
generate_with_adapter(base_model, dataset_occupancy)
generate_with_adapter(base_model, dataset_occupancy, target_rows=occupancy_test_len, forecasting=True)


# 3. Power Consumption
train_specific_adapter(base_model, dataset_power, strategy_get_prompting)
generate_with_adapter(base_model, dataset_power)
generate_with_adapter(base_model, dataset_power, target_rows=power_test_len, forecasting=True)

# 4. Traffic Volume
train_specific_adapter(base_model, dataset_traffic, strategy_get_prompting)
generate_with_adapter(base_model, dataset_traffic)
generate_with_adapter(base_model, dataset_traffic, target_rows=traffic_test_len, forecasting=True)